In [16]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import torch
import torchvision
from torch import nn
from torch.nn import functional as F
from torch.nn import CrossEntropyLoss
from torch.utils.data import Dataset, DataLoader, random_split
from torch.optim import SGD
from tensorflow.keras.datasets import mnist

In [17]:
import numpy as np
import torch
import torchvision
import matplotlib.pyplot as plt
from time import time
from torchvision import datasets, transforms
from torch import nn, optim

In [18]:
transform = transforms.Compose([transforms.ToTensor(),
                              transforms.Normalize((0.5,), (0.5,)),
                              ])

In [19]:
train_data = torchvision.datasets.MNIST(root = '/train', train = True, download = True, transform = transform)

In [20]:
test_data = torchvision.datasets.MNIST(root = './test', train = False, download = True, transform = transform)

In [21]:
valid_data, train_data = random_split(train_data, [48000,12000])

In [22]:
train_dataloader = DataLoader(train_data, batch_size = 120)
valid_dataloader = DataLoader(valid_data, batch_size = 120)
test_dataloader = DataLoader(test_data, batch_size = 120)

In [23]:
class WrappedDataLoader:
    def __init__(self, dl, func):
        self.dl = dl
        self.func = func

    def __len__(self):
        return len(self.dl)

    def __iter__(self):
        batches = iter(self.dl)
        for b in batches:
            yield (self.func(*b))

In [24]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def to_device(x, y):
    return x.to(device), y.to(device, dtype=torch.int64)

train_loader = WrappedDataLoader(train_dataloader, to_device)
valid_loader = WrappedDataLoader(valid_dataloader, to_device)
test_loader = WrappedDataLoader(test_dataloader, to_device)

In [25]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class ModelM7(nn.Module):
    def __init__(self):
        super(ModelM7, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, bias=False)       # output becomes 26x26
        self.conv1_bn = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 48, 3, bias=False)      # output becomes 24x24
        self.conv2_bn = nn.BatchNorm2d(48)
        self.conv3 = nn.Conv2d(48, 64, 3, bias=False)      # output becomes 22x22
        self.conv3_bn = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 80, 3, bias=False)      # output becomes 20x20
        self.conv4_bn = nn.BatchNorm2d(80)
        self.conv5 = nn.Conv2d(80, 96, 3, bias=False)      # output becomes 18x18
        self.conv5_bn = nn.BatchNorm2d(96)
        self.conv6 = nn.Conv2d(96, 112, 3, bias=False)     # output becomes 16x16
        self.conv6_bn = nn.BatchNorm2d(112)
        self.conv7 = nn.Conv2d(112, 128, 3, bias=False)    # output becomes 14x14
        self.conv7_bn = nn.BatchNorm2d(128)
        self.conv8 = nn.Conv2d(128, 144, 3, bias=False)    # output becomes 12x12
        self.conv8_bn = nn.BatchNorm2d(144)
        self.conv9 = nn.Conv2d(144, 160, 3, bias=False)    # output becomes 10x10
        self.conv9_bn = nn.BatchNorm2d(160)
        self.conv10 = nn.Conv2d(160, 176, 3, bias=False)   # output becomes 8x8
        self.conv10_bn = nn.BatchNorm2d(176)
        self.fc1 = nn.Linear(11264, 10, bias=False)
        self.fc1_bn = nn.BatchNorm1d(10)
    def get_logits(self, x):
        x = (x - 0.5) * 2.0
        conv1 = F.relu(self.conv1_bn(self.conv1(x)))
        conv2 = F.relu(self.conv2_bn(self.conv2(conv1)))
        conv3 = F.relu(self.conv3_bn(self.conv3(conv2)))
        conv4 = F.relu(self.conv4_bn(self.conv4(conv3)))
        conv5 = F.relu(self.conv5_bn(self.conv5(conv4)))
        conv6 = F.relu(self.conv6_bn(self.conv6(conv5)))
        conv7 = F.relu(self.conv7_bn(self.conv7(conv6)))
        conv8 = F.relu(self.conv8_bn(self.conv8(conv7)))
        conv9 = F.relu(self.conv9_bn(self.conv9(conv8)))
        conv10 = F.relu(self.conv10_bn(self.conv10(conv9)))
        flat1 = torch.flatten(conv10.permute(0, 2, 3, 1), 1)
        logits = self.fc1_bn(self.fc1(flat1))
        return logits
    def forward(self, x):
        logits = self.get_logits(x)
        return F.log_softmax(logits, dim=1)

In [26]:
model = ModelM7()

In [27]:
optimizer = optim.Adam(model.parameters(),lr=0.001)

In [28]:
criterion = torch.nn.CrossEntropyLoss()

In [29]:
def loss_batch(model, loss_func, x, y, opt=None):
    loss = loss_func(model(x), y)
    
    if opt is not None:
        loss.backward()
        opt.step()
        opt.zero_grad()

    return loss.item(), len(x)
def valid_batch(model, loss_func, x, y):
    output = model(x)
    loss = loss_func(output, y)
    pred = torch.argmax(output, dim=1)
    correct = pred == y.view(*pred.shape)
    
    return loss.item(), torch.sum(correct).item(), len(x)
def fit(epochs, model, loss_func, opt, train_dl, valid_dl):
    for epoch in range(epochs):
        # Train model
        model.train()
        losses, nums = zip(*[loss_batch(model, loss_func, x, y, opt) for x, y in train_dl])
        train_loss = np.sum(np.multiply(losses, nums)) / np.sum(nums)
        # Validation model
        model.eval()
        with torch.no_grad():
            losses, corrects, nums = zip(*[valid_batch(model, loss_func, x, y) for x, y in valid_dl])
            valid_loss = np.sum(np.multiply(losses, nums)) / np.sum(nums)
            valid_accuracy = np.sum(corrects) / np.sum(nums) * 100
            print(f"[Epoch {epoch+1}/{epochs}] "
                  f"Train loss: {train_loss:.6f}\t"
                  f"Validation loss: {valid_loss:.6f}\t",
                  f"Validation accruacy: {valid_accuracy:.3f}%")
def evaluate(model, loss_func, dl):
    model.eval()
    with torch.no_grad():
        losses, corrects, nums = zip(*[valid_batch(model, loss_func, x, y) for x, y in dl])
        test_loss = np.sum(np.multiply(losses, nums)) / np.sum(nums)
        test_accuracy = np.sum(corrects) / np.sum(nums) * 100
        
    print(f"Test loss: {test_loss:.6f}\t"
          f"Test accruacy: {test_accuracy:.3f}%")

In [30]:
n_epochs = 50
model = ModelM7().to(device)
optimizer = optim.Adam(model.parameters(),lr=0.001)
fit(n_epochs, model, criterion, optimizer, train_loader, valid_loader)

[Epoch 1/50] Train loss: 0.518253	Validation loss: 0.413374	 Validation accruacy: 97.502%
[Epoch 2/50] Train loss: 0.311000	Validation loss: 0.336896	 Validation accruacy: 98.410%
[Epoch 3/50] Train loss: 0.243113	Validation loss: 0.254016	 Validation accruacy: 98.598%
[Epoch 4/50] Train loss: 0.195504	Validation loss: 0.195968	 Validation accruacy: 98.662%
[Epoch 5/50] Train loss: 0.159961	Validation loss: 0.173911	 Validation accruacy: 98.650%
[Epoch 6/50] Train loss: 0.134879	Validation loss: 0.151915	 Validation accruacy: 98.642%
[Epoch 7/50] Train loss: 0.114939	Validation loss: 0.119239	 Validation accruacy: 98.906%
[Epoch 8/50] Train loss: 0.096488	Validation loss: 0.104495	 Validation accruacy: 98.890%
[Epoch 9/50] Train loss: 0.084055	Validation loss: 0.099410	 Validation accruacy: 98.821%
[Epoch 10/50] Train loss: 0.071538	Validation loss: 0.089193	 Validation accruacy: 98.925%
[Epoch 11/50] Train loss: 0.061019	Validation loss: 0.075273	 Validation accruacy: 99.019%
[Epoch 1

In [31]:
evaluate(model, criterion, test_loader)

Test loss: 0.023702	Test accruacy: 99.380%
